In [1]:
import pandas as pd
payments = pd.read_csv("../data/raw/olist_order_payments_dataset.csv")


In [2]:
payments.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 103886 entries, 0 to 103885
Data columns (total 5 columns):
 #   Column                Non-Null Count   Dtype  
---  ------                --------------   -----  
 0   order_id              103886 non-null  object 
 1   payment_sequential    103886 non-null  int64  
 2   payment_type          103886 non-null  object 
 3   payment_installments  103886 non-null  int64  
 4   payment_value         103886 non-null  float64
dtypes: float64(1), int64(2), object(2)
memory usage: 4.0+ MB


In [3]:
payments.duplicated().sum()

np.int64(0)

In [4]:
payments["payment_type"].value_counts(dropna=False)

payment_type
credit_card    76795
boleto         19784
voucher         5775
debit_card      1529
not_defined        3
Name: count, dtype: int64

In [5]:
payments[["payment_sequential", "payment_installments", "payment_value"]].describe()

,payment_sequential,payment_installments,payment_value
count,103886.000000,103886.000000,103886.000000
mean,1.092679,2.853349,154.100380
std,0.706584,2.687051,217.494064
min,1.000000,0.000000,0.000000
25%,1.000000,1.000000,56.790000
50%,1.000000,1.000000,100.000000
75%,1.000000,4.000000,171.837500
max,29.000000,24.000000,13664.080000


## Additional payment checks

One row is a payment record within an order. An order can have multiple payment records. Check the combination of order ID and payment sequence rather than expecting order ID alone to be unique.

In [6]:
payments.duplicated(subset=["order_id", "payment_sequential"]).sum()

np.int64(0)

In [7]:
(payments[["payment_sequential", "payment_installments", "payment_value"]] < 0).sum()

payment_sequential      0
payment_installments    0
payment_value           0
dtype: int64

### Zero values

Count and inspect zero payment amounts and zero installment counts. Zero is different from a missing value. These checks do not change any records.

In [8]:
print("Zero payment values:", (payments["payment_value"] == 0).sum())
print("Zero installment counts:", (payments["payment_installments"] == 0).sum())

Zero payment values: 9
Zero installment counts: 2


In [9]:
payments.loc[
    (payments["payment_value"] == 0)
    | (payments["payment_installments"] == 0)
]

,order_id,payment_sequential,payment_type,payment_installments,payment_value
19922,8bcbe01d44d147f901cd3192671144db,4,voucher,1,0.00
36822,fa65dad1b0e818e3ccc5cb0e39231352,14,voucher,1,0.00
43744,6ccb433e00daae1283ccc956189c82ae,4,voucher,1,0.00
46982,744bade1fcf9ff3f31d860ace076d422,2,credit_card,0,58.69
51280,4637ca194b6387e2d538dc89b124b0ee,1,not_defined,1,0.00
57411,00b1cb0320190ca0daa2c88b35206009,1,not_defined,1,0.00
62674,45ed6e85398a87c253db47c2d9f48216,3,voucher,1,0.00
77885,fa65dad1b0e818e3ccc5cb0e39231352,13,voucher,1,0.00
79014,1a57108394169c0b47d8f876acc9ba2d,2,credit_card,0,129.94
94427,c8c528189310eaa44a745b8d9d26908b,1,not_defined,1,0.00


### Largest payments

Inspect large amounts without assuming they are errors.

In [10]:
payments.nlargest(5, "payment_value")

,order_id,payment_sequential,payment_type,payment_installments,payment_value
52107,03caa2c082116e1d31e67e9ae3700499,1,credit_card,1,13664.08
34370,736e1922ae60d0d6a89247b851902527,1,boleto,1,7274.88
41419,0812eb902a67711a1cb742b3cdaa65ae,1,credit_card,8,6929.31
49581,fefacc66af859508bf1a7934eab1e97f,1,boleto,1,6922.21
85539,f5136e38d1a14a4dbd87dff67da82701,1,boleto,1,6726.66


## Findings and decision

- 103,886 payment records, with no missing values or exact duplicate rows.
- No duplicate combinations of `order_id` and `payment_sequential`.
- No negative payment values, installment counts, or sequence numbers.
- 9 zero-value payments: 6 vouchers and 3 records with payment type `not_defined`.
- 2 credit-card records have zero installments, with payment values of 58.69 and 129.94. The cause is unconfirmed.
- Retain all records and original values, including `not_defined`. Do not infer corrected installment counts.
- Payment sequences number separate payment records within an order. Installments split a payment over time. Do not multiply payment value by installments when calculating totals.
- The processed copy below contains the audited data without value changes.

In [11]:
payments_clean = payments.copy()
payments_clean.to_csv("../data/processed/payments_clean.csv", index=False)